# Import modules

In [ ]:
import datetime
import json
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from pyproj import Transformer

import geopandas as gpd
from IPython.display import clear_output, HTML
from matplotlib import colors
from matplotlib import rcParams

# Read discharge data

In [ ]:
discharge = pd.read_csv('dfw_outputs_first_test.csv', index_col=0)
discharge.index = pd.to_datetime(discharge.index, utc=True)
discharge = discharge.map(lambda x: max(x, 1e-5))

# Read flowline data

In [ ]:
flowlines = gpd.read_file('/Users/mdbartos/Documents/Data/nhf_1.1.3.gpkg', layer='flowpaths')
flowlines = flowlines.to_crs(epsg=5070)
flowlines = flowlines.drop_duplicates('fp_id').set_index('fp_id')
flowlines.index = flowlines.index.astype(int).astype(str)
flowlines['flow'] = pd.Series(pd.Series(flowlines.index).map(discharge.max(axis=0)).values, index=flowlines.index)

# Plot maximum discharge for each flowline

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
flowlines.plot(ax=ax, color='b', zorder=2, linewidth=0.2)
flowlines.plot(ax=ax, color='b', zorder=3, linewidth=2*np.log(1 + flowlines['flow'].values))

# Create animation by saving frames at each timestep

In [ ]:
timestamps = discharge.loc[:'2026-05-21 00:00:00'].index
for i, index in enumerate(timestamps):
    clear_output(wait=True)
    fig = plt.figure(figsize=(12,12))
    ax = plt.axes()
    plt.axis('off')

    current_discharge = discharge.loc[index]
    flowlines['flow'] = pd.Series(pd.Series(flowlines.index).map(current_discharge).values, index=flowlines.index)
    flowlines.plot(ax=ax, color='b', zorder=3, linewidth=0.2)
    flowlines.plot(ax=ax, color='b', zorder=3, linewidth=2*np.log(1 + flowlines['flow'].values))
    
    ax.annotate(f'{index}', (0.65, 0.95), xycoords='axes fraction', fontsize=14)
    plt.savefig(f'./dfw_discharge_anim/{i:04}.jpg', bbox_inches='tight')